# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`

In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

Downloaded model.py successfully.
Downloaded train_utils.py successfully.


In [3]:
from model import *
from train_utils import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=256,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

In [5]:
modelV1 = CustomGPT(config)

if torch.cuda.device_count() > 1:
    modelV1 = nn.DataParallel(modelV1)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV1.to(device)

calc_params(modelV1)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs
Total Parameters: 30,586,449
Trainable Parameters: 30,586,449


# The Dataset

In [6]:
from datasets import load_dataset
from tqdm.auto import tqdm
target_tokens = 100_000_000
ds = load_dataset("HuggingFaceFW/fineweb", split="train", name="sample-10BT", streaming=True)

tokens = []
pbar = tqdm(total=target_tokens)
for sample in ds:
    text = sample["text"]
    tokenized = tokenizer.encode(text)
    tokens.extend(tokenized)
    curr = min(len(tokens), target_tokens)
    pbar.n = curr
    pbar.update(0)
    if len(tokens) >= target_tokens:
        break

pbar.close()
print(f"Collected {len(tokens)} tokens.")


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

  0%|          | 0/100000000 [00:00<?, ?it/s]

Collected 100021689 tokens.


In [7]:
data = torch.tensor(tokens, dtype=torch.long)

train_split = 0.9
train_dataloader, test_dataloader = create_dataloaders(tokens, train_split, device, block_size = config.block_size, batch_size=32)

Data length: 100021689
Train data length: 90019520
Test data length: 10002169
Train batches: 32.00009100274359
Test batches: 32.00079664704377


# Pretraining the model

In [8]:
from tqdm.auto import tqdm
steps = 61000
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
    modelV1.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    
    if step % 10000 == 0:
        modelV1.eval()
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {train_loss} | Test Loss: {test_loss:}")

  0%|          | 0/61000 [00:00<?, ?it/s]

0it [00:00, ?it/s]

Step 10000 | Train Loss: 5.710817337036133 | Test Loss: 5.48599100112915


0it [00:00, ?it/s]

Step 20000 | Train Loss: 5.379891395568848 | Test Loss: 5.108893418312073


...

Step 40000 | Train Loss: 4.993866443634033 | Test Loss: 4.763564848899842


0it [00:00, ?it/s]

Step 50000 | Train Loss: 4.8110551834106445 | Test Loss: 4.668863010406494


0it [00:00, ?it/s]

Step 60000 | Train Loss: 4.740047931671143 | Test Loss: 4.584244513511658


In [9]:
torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")

# Testing



In [10]:
test_model = CustomGPT(config).to(device)
state_dict = torch.load("tara_n1_pretrain_v1.pth", map_location=device)
state_dict = {k.removeprefix('module.'): v for k, v in state_dict.items()} 
test_model.load_state_dict(state_dict)

<All keys matched successfully>

In [11]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

test_model.eval()
with torch.inference_mode():
    output = test_model.generate(context, max_new_tokens=100)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")


Input:
Once upon a time, 

Output:
Once upon a time, 0: “A significant drop relies during the failure to jump into the bank. And fortunately the initial announcement of the loan waslimit., I saw a back injury report by Friday Morning Intairnary to discuss a real deal, in here his case.”
At the XXPHOption Basically, Jones, executive director at the Financial Berkshire Institute of Toledo, Montgomery, is an assistant forMail and a head star at executive level registrar.com, D.C., Richard McGek
